# 05. Grounded Rationale 학습 및 최종화

최종 score를 먼저 고정한 뒤 genuine OOF score로 silver를 만들고 rationale adapter를
학습한다. score는 rationale 단계에서 절대 수정하지 않으며, adapter가 승격되지 않으면
deterministic exact-evidence fallback으로 최종 스키마를 만든다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 파라미터와 실행 스위치


In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"

# 04에서 실제 승격된 하나의 score branch를 선택한다.
SELECTED_SCORE_BRANCH = "two_source"  # "two_source" 또는 "multisource"
score_branches = {
    "two_source": {
        "model": f"{RUN_NAMESPACE}_qwen_tfidf",
        "oof": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_qwen_tfidf_crossfit.jsonl",
        "target": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_stacked_validation.jsonl",
    },
    "multisource": {
        "model": f"{RUN_NAMESPACE}_multisource",
        "oof": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_crossfit.jsonl",
        "target": PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_validation.jsonl",
    },
}
if SELECTED_SCORE_BRANCH not in score_branches:
    raise ValueError(f"지원하지 않는 score branch: {SELECTED_SCORE_BRANCH}")
selected_scores = score_branches[SELECTED_SCORE_BRANCH]
SCORE_MODEL = selected_scores["model"]
OOF_SCORES = selected_scores["oof"]
TARGET_SCORES = selected_scores["target"]

RUN_SILVER = False
RUN_RATIONALE_TRAINING = False
RUN_ADAPTER_GENERATION = False
RUN_ADAPTER_FINALIZATION = False
RUN_FALLBACK_FINALIZATION = False
RUN_FINAL_SCHEMA_EVALUATION = False
RUN_BLIND_REVIEW_PACK = False
RUN_LOCAL_GGUF_JUDGE = False
RUN_JUDGE_SUMMARY = False
INSTALL_LOCAL_JUDGE_DEPENDENCY = False
FINAL_SCHEMA_VARIANT = "fallback"  # "fallback" 또는 "adapter"
if FINAL_SCHEMA_VARIANT not in {"fallback", "adapter"}:
    raise ValueError(f"지원하지 않는 final schema variant: {FINAL_SCHEMA_VARIANT}")

GPU_ENABLED = RUN_SILVER or RUN_RATIONALE_TRAINING or RUN_ADAPTER_GENERATION
require_model_revision(MODEL_REVISION, enabled=GPU_ENABLED)
require_bf16_gpu(enabled=GPU_ENABLED, recommended_vram_gb=35)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


## 3. genuine OOF 기반 grounded silver


In [ ]:
SILVER = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_silver_accepted.jsonl"
SILVER_REJECTED = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_silver_rejected.jsonl"
SILVER_REPORT = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_rationale_silver.json"

run_cli(
    "build grounded silver",
    RUN_SILVER,
    "scripts/build_silver_rationales.py",
    "--config", "configs/rationale_sft.yaml",
    "--input", TRAIN_PATH,
    "--scores", OOF_SCORES,
    "--folds", FOLDS_PATH,
    "--model-revision", MODEL_REVISION,
    "--output", SILVER,
    "--rejected-output", SILVER_REJECTED,
    "--report", SILVER_REPORT,
    *download_flag,
)
silver_report = show_json(SILVER_REPORT)
if RUN_RATIONALE_TRAINING:
    if not silver_report or not silver_report.get("promotion_eligible", False):
        raise RuntimeError("silver acceptance gate를 통과하지 못했습니다.")


## 4. rationale QLoRA SFT


In [ ]:
RATIONALE_RUN_ID = f"rationale_{RUN_NAMESPACE}"
RATIONALE_MODEL_DIR = PROJECT_ROOT / "artifacts" / "models" / RATIONALE_RUN_ID

run_cli(
    "train rationale adapter",
    RUN_RATIONALE_TRAINING,
    "-m", "src.train.train_rationale",
    "--config", "configs/rationale_sft.yaml",
    "--input", TRAIN_PATH,
    "--silver", SILVER,
    "--run-id", RATIONALE_RUN_ID,
    "--model-revision", MODEL_REVISION,
    *download_flag,
)


## 5. adapter rationale 생성과 최종 스키마


In [ ]:
TARGET_INPUT = VALIDATION_PATH  # 최종 test에서는 TEST_PATH로 바꾼다.
RATIONALE_OUTPUT = PROJECT_ROOT / "artifacts" / "rationale" / f"{RUN_NAMESPACE}_target_grounded.jsonl"
RATIONALE_REPORT = PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_target_rationale.json"
ADAPTER_FINAL = PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_validation.jsonl"
FALLBACK_FINAL = PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_validation.jsonl"

run_cli(
    "generate adapter rationales",
    RUN_ADAPTER_GENERATION,
    "scripts/generate_rationales.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--checkpoint", RATIONALE_MODEL_DIR / "best.json",
    "--precision", "4bit",
    "--max-attempts", "2",
    "--output", RATIONALE_OUTPUT,
    "--report", RATIONALE_REPORT,
    *download_flag,
)
run_cli(
    "adapter finalization",
    RUN_ADAPTER_FINALIZATION,
    "scripts/finalize_predictions.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--rationales", RATIONALE_OUTPUT,
    "--output", ADAPTER_FINAL,
    "--ledger", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_ledger.jsonl",
    "--bare-output", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_adapter_submission.jsonl",
    "--model-name", f"{SCORE_MODEL}_adapter_rationale",
)
run_cli(
    "deterministic fallback finalization",
    RUN_FALLBACK_FINALIZATION,
    "scripts/finalize_predictions.py",
    "--input", TARGET_INPUT,
    "--scores", TARGET_SCORES,
    "--output", FALLBACK_FINAL,
    "--ledger", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_ledger.jsonl",
    "--bare-output", PROJECT_ROOT / "artifacts" / "final" / f"{RUN_NAMESPACE}_fallback_submission.jsonl",
    "--model-name", f"{SCORE_MODEL}_deterministic_fallback",
)


## 6. final schema 평가 및 blind review


In [ ]:
run_cli(
    "final schema evaluation",
    RUN_FINAL_SCHEMA_EVALUATION,
    "scripts/evaluate_predictions.py",
    "--gold", VALIDATION_PATH,
    "--pred", {"fallback": FALLBACK_FINAL, "adapter": ADAPTER_FINAL}[FINAL_SCHEMA_VARIANT],
    "--final-schema",
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_final_schema.json",
)

REVIEW_PACK = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_blind_100.jsonl"
REVIEW_KEY = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_blind_100_key.jsonl"
run_cli(
    "blind review pack",
    RUN_BLIND_REVIEW_PACK,
    "scripts/build_rationale_review_pack.py",
    "--input", VALIDATION_PATH,
    "--candidate", ADAPTER_FINAL,
    "--baseline", FALLBACK_FINAL,
    "--sample-size", "100",
    "--seed", "2026",
    "--output", REVIEW_PACK,
    "--key-output", REVIEW_KEY,
)


## 7. 선택적 로컬 GGUF judge


In [ ]:
GGUF_MODEL = Path("/content/drive/MyDrive/models/pinned-rationale-judge.Q4_K_M.gguf")
JUDGMENTS = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_judgments.jsonl"
JUDGE_SUMMARY = PROJECT_ROOT / "artifacts" / "reviews" / f"{RUN_NAMESPACE}_judge_summary.json"

if INSTALL_LOCAL_JUDGE_DEPENDENCY:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[judge]"],
        cwd=PROJECT_ROOT,
    )

run_cli(
    "local GGUF judge",
    RUN_LOCAL_GGUF_JUDGE,
    "scripts/judge_rationales_local.py",
    "--config", "configs/rationale_judge.yaml",
    "--review-pack", REVIEW_PACK,
    "--model", GGUF_MODEL,
    "--output", JUDGMENTS,
)
run_cli(
    "summarize judge after judging",
    RUN_JUDGE_SUMMARY,
    "scripts/summarize_rationale_judge.py",
    "--judgments", JUDGMENTS,
    "--key", REVIEW_KEY,
    "--output", JUDGE_SUMMARY,
)


## 8. 승격 규칙

adapter와 fallback은 동일한 score 파일을 사용해야 한다. 사람 blind 평가에서 adapter가
우세할 때만 승격하며 자동 judge는 보조 신호일 뿐이다. 어느 경로든 ledger는 내부 감사용이고
bare submission에는 포함하지 않는다.
